<a href="https://colab.research.google.com/github/KoraRiko/joke_generator/blob/main/AI_Cleaning_and_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing libraries

In [ ]:
!pip install yake pandas requests --quiet

Loading datasets

In [ ]:
import requests, json, pandas as pd

# Reddit jokes (195K jokes)
url = "https://raw.githubusercontent.com/taivop/joke-dataset/master/reddit_jokes.json"
reddit = requests.get(url).json()
df_reddit = pd.DataFrame(reddit)[["title", "body", "score"]]
df_reddit["joke"] = df_reddit["title"] + " " + df_reddit["body"]
df_reddit = df_reddit[["joke", "score"]]

# Wocka jokes (10K jokes, have categorys)
url2 = "https://raw.githubusercontent.com/taivop/joke-dataset/master/wocka.json"
wocka = requests.get(url2).json()
df_wocka = pd.DataFrame(wocka)[["body", "category"]]
df_wocka.columns = ["joke", "category"]
df_wocka["score"] = 5

print(f"Reddit: {len(df_reddit)} jokes")
print(f"Wocka:  {len(df_wocka)} jokes")

Reddit: 194553 шуток
Wocka:  10019 шуток


In [ ]:
import pandas as pd

# Reading the Kaggle dataset
df_kaggle = pd.read_csv("train.csv")
print("Columns in the file", df_kaggle.columns.tolist())
df_kaggle.head(3)

Колонки в файле: ['Unnamed: 0', 'joke']


,Unnamed: 0,joke
0,0,A steak pun is a rare medium well done.
1,1,They say that breakfast is the most important ...
2,2,What do you get if you cross an angry sheep wi...


Combine all three sources

In [ ]:

df_kaggle["score"] = 5
print(f"Kaggle: {len(df_kaggle)} jokes")

df_all= pd.concat([
    df_reddit[["joke", "score"]],   # GitHub Reddit
    df_wocka[["joke", "score"]],    # GitHub Wocka
    df_kaggle[["joke", "score"]]   # Kaggle Short Jokes
], ignore_index=True)

print(f"Before cleaning: {len(df_all)} jokes")

Data cleaning

In [ ]:
from transformers import pipeline
from tqdm import tqdm

# Load both models
humor_pipe = pipeline(
    "text-classification",
    model="likhithasapu/humour-detection-mBert",
    truncation=True, max_length=128
)
toxic_pipe = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    truncation=True, max_length=128
)

df = df_all.copy()

#STEP 1 - basic cleaning
df = df[df["joke"].str.len() > 20]
df = df[df["joke"].str.len() < 500]
df = df.drop_duplicates(subset="joke")
print(f"After basic cleansing: {len(df)}")

#STEP 2 - Bad Words Filter
bad_words = [
    "anal", "rape", "porn", "penis", "vagina",
    "hitler", "nazi", "nigger", "racist",
    "trump", "clinton", "obama", "political",
    "[removed]", "[deleted]", "Putin", "nsfw",
]
mask = df["joke"].str.lower().apply(
    lambda x: not any(word in x for word in bad_words)
)
df = df[mask]
print(f"After the bad words filter: {len(df)}")

#STEP 3 - toxicity filter
toxic_scores = []
for i in tqdm(range(0, len(df), 64)):
    batch = df["joke"].iloc[i:i+64].tolist()
    results = toxic_pipe(batch)
    toxic_scores.extend([r["score"] for r in results])

df["toxic_score"] = toxic_scores
THRESHOLD = 0.3
df = df[df["toxic_score"] < THRESHOLD]
print(f"After the toxicity filter: {len(df)}")

#STEP 4 - Select the top results by score
df = df.nlargest(30000, "score").reset_index(drop=True)

#STEP 5 - humour classifier
print("StartIing humor classifier...")
labels = []
for i in tqdm(range(0, len(df), 64)):
    batch = df["joke"].iloc[i:i+64].tolist()
    results = humor_pipe(batch)
    labels.extend([r["label"] for r in results])

df["is_humor"] = labels
df = df[df["is_humor"] == "Humour"]
df = df.nlargest(20000, "score").reset_index(drop=True)
print(f"Total number of clean jokes: {len(df)}")
df.to_csv("cleaned_jokes.csv", index=False, encoding="utf-8-sig")
print("The dataset has been successfully saved to cleaned_jokes.csv")


In [ ]:
print(f"Total number of clean jokes: {len(df)}")
df.head(20)

Keyword extraction

In [ ]:
import yake

kw_extractor = yake.KeywordExtractor(lan="en", n=1, top=5)

def extract_keywords(joke):
    keywords = kw_extractor.extract_keywords(joke)

    scored = []
    for kw, _ in keywords:
        score = similarity(joke, kw)
        scored.append((kw.lower(), score))

    #sorted by relevance
    scored.sort(key=lambda x: x[1], reverse=True)

    #use only one single best keyword
    if scored and scored[0][1] > 0.5:
        return scored[0][0]

    return "general"

Keyword check

In [ ]:
print(f"Jokes in df: {len(df)}")
df[["keyword", "joke"]].head(20)

Creating JSONL for fine-tuning


In [ ]:
import json

SYSTEM_PROMPT = "You are a funny assistant. Generate a short, clever joke based on the given keyword."

with open("jokes_finetune_2.jsonl", "w") as f:
    for _, row in df.iterrows():
        entry = {
            "messages": [
                {"role": "system",  "content": SYSTEM_PROMPT},
                {"role": "user",    "content": f"Generate a joke about: {row['keyword']}"},
                {"role": "assistant", "content": row["joke"]}
            ]
        }
        f.write(json.dumps(entry) + "\n")

print(f"File created: {len(df)} examples")

print("\nFile check:")
with open("jokes_finetune_2.jsonl", "r") as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(json.loads(line)["messages"][1]["content"])  # keyword
        print(json.loads(line)["messages"][2]["content"][:60])
        print("---")

Fine-tune

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
# Upload the file to OpenAI
with open("jokes_finetune_2.jsonl", "rb") as f:
    response = client.files.create(file=f, purpose="fine-tune")
file_id = response.id

#Start the training
job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18"
)
print(f"Job ID: {job.id}")
print("The training has begun")

Job ID: ftjob-YkNWhKg0kbuIwSdRf3bjiKXl
Обучение запущено! ☕


File verification

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

#Download file from OpenAI
file_id = "..."
content = client.files.content(file_id)

#Save in Colab
with open("jokes_finetune_2.jsonl", "wb") as f:
    f.write(content.content)

#Checking
import json
with open("jokes_finetune_2.jsonl", "r") as f:
    for i, line in enumerate(f):
        if i >= 20: break
        entry = json.loads(line)
        keyword = entry["messages"][1]["content"].replace("Generate a joke about: ", "")
        print(f"{i+1}. [{keyword}]")